# CBF 최종 공통 평가 — Google Colab

**공통 Train → Validation에서 설정 선택 → Train+Validation → 공통 Test 평가 1회**

알고리즘 본체는 `cbf/run_cbf.py`·`cbf_helpers.py`에 구현되어 있다. 이 노트북은 동일한 코드를 Colab에서 실행하고 결과를 확인한다.

## 1. 실행 파일 및 공통 데이터 업로드

Colab 왼쪽 **파일** 메뉴에서 다음 6개 파일을 `/content`에 올리거나, 다음 셀의 업로드 창을 사용한다.

- `CBF_Final_Colab_Kit.zip` (CBF `.py` + 팀 공통 `common/` 스크립트)
- `team_split.csv`
- `contraindication_conflicts.csv`
- `team_eligible_indications.csv`
- `disease_gene_t03.csv`
- `drug_gene.csv`

공통 분할은 새로 생성하지 않는다.

In [ ]:
from pathlib import Path
import datetime
import json
import os
import subprocess
import sys
import zipfile

ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
REQUIRED = [
    'CBF_Final_Colab_Kit.zip', 'team_split.csv',
    'contraindication_conflicts.csv', 'team_eligible_indications.csv',
    'disease_gene_t03.csv', 'drug_gene.csv',
]
missing = [name for name in REQUIRED if not (ROOT / name).is_file()]
if missing:
    try:
        from google.colab import files
        print('다음 파일을 업로드하세요:', ', '.join(missing))
        files.upload()
    except ImportError:
        raise FileNotFoundError('필요한 파일을 현재 폴더에 배치하세요: ' + ', '.join(missing))
remaining = [name for name in REQUIRED if not (ROOT / name).is_file()]
if remaining:
    raise FileNotFoundError('파일이 없습니다: ' + ', '.join(remaining))

STAMP = datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
PROJECT = ROOT / ('cbf_project_' + STAMP)
PROJECT.mkdir()
with zipfile.ZipFile(ROOT / 'CBF_Final_Colab_Kit.zip') as archive:
    entries = archive.namelist()
    for name in entries:
        entry = Path(name)
        if name.startswith('/') or '..' in entry.parts or entry.parts[0] not in {'cbf','common'}:
            raise ValueError('압축 파일 경로가 올바르지 않습니다: ' + name)
    archive.extractall(PROJECT)
assert (PROJECT/'cbf/run_cbf.py').is_file()
assert (PROJECT/'common/evaluate.py').is_file()
assert (PROJECT/'common/prepare_features.py').is_file()
print('프로젝트:', PROJECT)

## 2. 환경 설정

`requirements.txt`의 버전을 사용한다. 계산 코드는 별도 Python 프로세스에서 실행된다.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(PROJECT / 'cbf/requirements.txt')
], check=True)
print('라이브러리 설치 완료')

## 3. 공통 데이터 준비 및 입력 검증

분할·금기 파일의 SHA-256을 확인하고, 질병–유전자 열을 표준화한다.

In [ ]:
PREPARED = ROOT / ('cbf_prepared_' + STAMP)
subprocess.run([
    sys.executable, str(PROJECT/'cbf/prepare_inputs.py'),
    '--source-dir', str(ROOT),
    '--output-dir', str(PREPARED)
], check=True)
print(json.dumps(json.loads((PREPARED/'verified_hashes.json').read_text()), indent=2))

## 4. 공통 Validation 설정 선택 → Test 1회 평가

12개 설정(Raw/IDF × k=10·20·50 × p=1·2) 중 Validation NDCG@10 최대 설정을 사용한다. 추천 결과는 팀의 `common/evaluate.py`로 계산한다.

In [ ]:
RESULTS = ROOT / ('cbf_run_' + STAMP)
env = dict(os.environ, OPENBLAS_NUM_THREADS='2', OMP_NUM_THREADS='2')
subprocess.run([
    sys.executable, str(PROJECT/'cbf/run_cbf.py'),
    '--data-dir', str(PREPARED),
    '--output-dir', str(RESULTS)
], check=True, env=env)
summary = json.loads((RESULTS/'results/test/cbf_summary.json').read_text())
setting = json.loads((RESULTS/'results/selected_config.json').read_text())
print('선정된 설정:', setting)
for key in ('recall@10','ndcg@10','recall@50','recall@100','query_coverage','catalog_coverage@10'):
    print(f'{key}: {summary[key]:.6f}')
print('최종 Test JSON:', RESULTS/'results/test/cbf_summary.json')

## 5. 결과 확인 및 다운로드

저장소에 제출하는 것과 동일한 `results/` 집계 구조만 다운로드한다. 원본 데이터와 추천 점수 전체 CSV는 ZIP에 포함하지 않는다.

In [ ]:
import pandas as pd
from IPython.display import display, Image

display(pd.read_csv(RESULTS/'results/validation_comparison.csv'))
display(Image(filename=str(RESULTS/'results/cbf_validation_sensitivity.png')))
print('Test 주요 지표:')
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
SHARE_ZIP = ROOT / ('CBF_Final_Summary_' + STAMP + '.zip')
with zipfile.ZipFile(SHARE_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted((RESULTS/'results').rglob('*')):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(RESULTS))
print('공유용 결과 파일:', SHARE_ZIP)
try:
    from google.colab import files
    files.download(str(SHARE_ZIP))
except ImportError:
    pass